In [ ]:
import torch
import torch.nn as nn
from typing import Tuple

from cs590_env.schema import (
    MAX_HAND_SIZE,
    MAX_JOKER_DISPLAY,
    NUM_HAND_TYPES,
    NUM_RANKS,
    NUM_SUITS,
    NUM_BOSS_BLINDS,
)
from balatro_gym.cards import Enhancement, Edition, Seal
from balatro_gym.jokers import JOKER_LIBRARY

In [ ]:
class CardEmbedding(nn.Module):
    """Embed a sequence of playing cards via additive composition:
    rank + suit + enhancement + edition + seal.

    Each instance owns its own embedding tables, so hand cards and
    deck cards can learn distinct representations for the same card.
    """

    _NUM_RANKS        = NUM_RANKS              # 13: TWO(0) .. ACE(12)
    _NUM_SUITS        = NUM_SUITS              # 4:  CLUBS(0) .. SPADES(3)
    _NUM_ENHANCEMENTS = len(Enhancement)       # 9:  NONE(0) .. LUCKY(8)
    _NUM_EDITIONS     = len(Edition)           # 5:  NONE(0) .. NEGATIVE(4)
    _NUM_SEALS        = len(Seal)              # 5:  NONE(0) .. PURPLE(4)

    def __init__(self, d_model: int):
        super().__init__()
        D = d_model
        self.rank_emb        = nn.Embedding(self._NUM_RANKS, D)
        self.suit_emb        = nn.Embedding(self._NUM_SUITS, D)
        self.enhancement_emb = nn.Embedding(self._NUM_ENHANCEMENTS, D)
        self.edition_emb     = nn.Embedding(self._NUM_EDITIONS, D)
        self.seal_emb        = nn.Embedding(self._NUM_SEALS, D)

    def forward(
        self,
        card_ids: torch.Tensor,        # (B, N) long, -1 = empty
        enhancements: torch.Tensor,    # (B, N) long
        editions: torch.Tensor,        # (B, N) long
        seals: torch.Tensor,           # (B, N) long
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns:
            toks: (B, N, D) — padding positions zeroed out
            mask: (B, N)    — True for real cards
        """
        mask = card_ids >= 0
        safe = card_ids.clamp(min=0)

        toks = (
            self.rank_emb(safe // 4)
            + self.suit_emb(safe % 4)
            + self.enhancement_emb(enhancements)
            + self.edition_emb(editions)
            + self.seal_emb(seals)
        )
        toks = toks * mask.unsqueeze(-1).float()
        return toks, mask

In [ ]:
class CombatEmbeddings(nn.Module):
    """Embed raw combat observations into four token sequences for the
    transformer backbone.

    Input groups (keys from BalatroPhaseWrapper combat obs):
        Hand cards:   hand_card_{ids,enhancements,editions,seals}, flag arrays
        Deck cards:   deck_card_{ids,enhancements,editions,seals}
        Combat state: hands/discards remaining, money, scores, hand levels
        Modifiers:    boss blind id + active flag, joker ids

    Hand and deck cards each have their own CardEmbedding instance
    (separate weights) so the model can learn distinct representations
    for "card in hand" vs "card in draw pile".  Hand cards additionally
    receive a flag projection for face_down / debuffed.

    Returns:
        hand_toks   (B, MAX_HAND, D)       per-hand-card embeddings
        hand_mask   (B, MAX_HAND)           True for real hand cards
        deck_toks   (B, MAX_DECK, D)        per-deck-card embeddings
        deck_mask   (B, MAX_DECK)           True for real deck cards
        combat_seq  (B, 13, D)              [run_tok, 12× hand_level_tok]
        mod_seq     (B, 1+MAX_JOKERS, D)    [boss_tok, joker_tok × MAX_JOKERS]
        mod_mask    (B, 1+MAX_JOKERS)       True for real modifiers
    """

    # ── Vocabulary sizes (derived from game definitions) ────────
    _NUM_HAND_TYPES  = NUM_HAND_TYPES                      # 12
    _NUM_BOSS_IDS    = NUM_BOSS_BLINDS + 1                 # 29 (0 = no boss, 1-28 = boss types)
    _NUM_JOKER_IDS   = len(JOKER_LIBRARY) + 1              # 151 (0 = padding, 1-150 = joker ids)
    _MAX_HAND        = MAX_HAND_SIZE                       # 8
    _MAX_DECK        = NUM_RANKS * NUM_SUITS               # 52
    _MAX_JOKERS      = MAX_JOKER_DISPLAY                   # 10

    # ── Feature vector widths (structural choices of this layer) ─
    NUM_RUN_SCALARS  = 5      # hands_rem, disc_rem, money,
                              #   cur_score, target_score
    NUM_HAND_FLAGS   = 2      # face_down, debuffed (hand-only)
    HL_FEATS         = 3      # level, base_chips, base_mult

    def __init__(self, d_model: int = 128):
        super().__init__()
        D = d_model

        # ── Card embeddings (separate weights per context) ───────
        self.hand_card_emb = CardEmbedding(D)
        self.deck_card_emb = CardEmbedding(D)

        # ── Hand-only flag projection ────────────────────────────
        self.hand_flags_proj = nn.Linear(self.NUM_HAND_FLAGS, D, bias=False)

        # ── Run-state scalar → single token ──────────────────────
        self.run_proj = nn.Linear(self.NUM_RUN_SCALARS, D)

        # ── Hand-level → 12 tokens (type emb + stat proj, summed) ─
        self.hand_type_emb   = nn.Embedding(self._NUM_HAND_TYPES, D)
        self.hand_level_proj = nn.Linear(self.HL_FEATS, D)

        # ── Boss → single token ──────────────────────────────────
        self.boss_emb         = nn.Embedding(self._NUM_BOSS_IDS, D)
        self.boss_active_proj = nn.Linear(1, D, bias=False)

        # ── Joker → up to MAX_JOKERS tokens ──────────────────────
        self.joker_emb = nn.Embedding(self._NUM_JOKER_IDS, D, padding_idx=0)

        # ── LayerNorm per output group ───────────────────────────
        self.hand_ln   = nn.LayerNorm(D)
        self.deck_ln   = nn.LayerNorm(D)
        self.combat_ln = nn.LayerNorm(D)
        self.mod_ln    = nn.LayerNorm(D)

    # ------------------------------------------------------------------

    @staticmethod
    def _signed_log1p(x: torch.Tensor) -> torch.Tensor:
        """sign(x) * log(1 + |x|)  — compresses large magnitudes while
        preserving sign (money can go negative with Credit Card joker)."""
        return x.sign() * torch.log1p(x.abs())

    # ------------------------------------------------------------------

    def forward(self, obs: dict):
        device = obs['hand_card_ids'].device
        B = obs['hand_card_ids'].shape[0]

        # ── 1. Hand card tokens ──────────────────────────────────
        hand_toks, hand_mask = self.hand_card_emb(
            obs['hand_card_ids'].long(),
            obs['hand_card_enhancements'].long(),
            obs['hand_card_editions'].long(),
            obs['hand_card_seals'].long(),
        )
        flags = torch.stack([
            obs['hand_is_face_down'].float(),
            obs['hand_is_debuffed'].float(),
        ], dim=-1)                                                 # (B, 8, 2)
        hand_toks = hand_toks + self.hand_flags_proj(flags) * hand_mask.unsqueeze(-1).float()
        hand_toks = self.hand_ln(hand_toks)                        # (B, 8, D)

        # ── 2. Deck card tokens (own weights, no flags) ─────────
        deck_toks, deck_mask = self.deck_card_emb(
            obs['deck_card_ids'].long(),
            obs['deck_card_enhancements'].long(),
            obs['deck_card_editions'].long(),
            obs['deck_card_seals'].long(),
        )
        deck_toks = self.deck_ln(deck_toks)                        # (B, ≤52, D)

        # ── 3. Run-state token ───────────────────────────────────
        run_feats = torch.stack([
            obs['hands_remaining'].float(),
            obs['discards_remaining'].float(),
            self._signed_log1p(obs['money'].float()),
            torch.log1p(obs['current_score'].float()),
            torch.log1p(obs['target_score'].float()),
        ], dim=-1)                                                 # (B, 5)
        run_tok = self.run_proj(run_feats).unsqueeze(1)            # (B, 1, D)

        # ── 4. Hand-level tokens (12) ────────────────────────────
        hl_feats = obs['hand_levels'].float()                      # (B, 12, 3)
        ht_idx   = torch.arange(self._NUM_HAND_TYPES, device=device)
        hl_toks  = (self.hand_type_emb(ht_idx)                     # (12, D) broadcast
                    + self.hand_level_proj(hl_feats))               # (B, 12, D)

        combat_seq = torch.cat([run_tok, hl_toks], dim=1)
        combat_seq = self.combat_ln(combat_seq)                    # (B, 13, D)

        # ── 5. Boss token ────────────────────────────────────────
        boss_tok = (
            self.boss_emb(obs['boss_id'].long())
            + self.boss_active_proj(obs['boss_is_active'].float().unsqueeze(-1))
        ).unsqueeze(1)                                             # (B, 1, D)

        # ── 6. Joker tokens ──────────────────────────────────────
        joker_toks = self.joker_emb(obs['joker_ids'].long())       # (B, 10, D)

        mod_seq = torch.cat([boss_tok, joker_toks], dim=1)
        mod_seq = self.mod_ln(mod_seq)                             # (B, 11, D)

        boss_real  = torch.ones(B, 1, dtype=torch.bool, device=device)
        joker_real = obs['joker_is_empty'].long() == 0             # (B, 10)
        mod_mask   = torch.cat([boss_real, joker_real], dim=1)     # (B, 11)

        return hand_toks, hand_mask, deck_toks, deck_mask, combat_seq, mod_seq, mod_mask

In [ ]:
class CombatBackbone(nn.Module):

In [ ]:
class CombatHeads(nn.Module):

In [ ]:
class CombatPPOAgent(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.embeddings = CombatEmbeddings(d_model=d_model)
        self.backbone = CombatBackbone(d_model=d_model)
        self.heads = CombatHeads(d_model=d_model)

    def forward(self, s):
        """
        Maps a raw state dictionary 's' to action logits and a value estimate.

        Args:
            s (dict): Raw feature dictionary (e.g., {'run_token': {'money': tensor(...)}} )

        Returns:
            selection_logits: (Batch, 8, 2)
            execution_logits: (Batch, 2)
            state_value: (Batch, 1)
        """

        # 1. Embeddings
        card_toks, combat_seq, mod_seq = self.embeddings(s)

        # 2. Backbone
        card_feats, global_ctx = self.backbone(card_toks, combat_seq, mod_seq)

        # 3. Heads
        sel_logits, exec_logits, value = self.heads(card_feats, global_ctx)

        return sel_logits, exec_logits, value